In [1]:
import os
os.chdir(r"D:\Projects\Poverty Predictor Bd")

import pandas as pd
import numpy as np
import joblib
import shap

# Load model and data
df       = pd.read_csv("data/processed/master_features.csv")
model    = joblib.load("models/random_forest_final.pkl")
scaler   = joblib.load("models/scaler_final.pkl")
features = joblib.load("models/feature_cols.pkl")

X = df[features]
X_scaled = scaler.transform(X)

# Compute SHAP values
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_scaled)

# Save SHAP values as dataframe
shap_df = pd.DataFrame(shap_values, columns=features)
shap_df.insert(0, 'district_name', df['district_name'].values)
shap_df.to_csv("data/processed/shap_values.csv", index=False)

# Save model predictions
df['poverty_predicted'] = model.predict(X_scaled)
df.to_csv("data/processed/master_features.csv", index=False)

print(f"SHAP values saved: {shap_df.shape}")
print(f"Predictions added to master_features.csv")
print(f"\nTop features by mean |SHAP|:")
mean_shap = np.abs(shap_values).mean(axis=0)
for feat, val in sorted(zip(features, mean_shap), 
                         key=lambda x: -x[1]):
    print(f"  {feat:<35} {val:.4f}")

SHAP values saved: (64, 20)
Predictions added to master_features.csv

Top features by mean |SHAP|:
  elevation_mean                      0.8445
  ntl_mean_spatial_lag                0.6016
  road_density                        0.4424
  ntl_iqr                             0.4329
  ntl_yoy_change                      0.3487
  ndvi_mean_spatial_lag               0.3421
  ntl_per_capita_spatial_lag          0.1545
  water_fraction                      0.1361
  ntl_max                             0.1113
  ndvi_std                            0.0972
  ntl_trend                           0.0950
  pop_density_spatial_lag             0.0906
  pop_density                         0.0832
  elevation_std                       0.0817
  ntl_mean                            0.0792
  ntl_std                             0.0659
  urban_fraction                      0.0615
  ndvi_mean                           0.0574
  ntl_per_capita                      0.0545
